# This is the start of the coursework.

In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


### Data Preparation

### Loading and Inspecting Data
We first begin by importing the training dataset (`epl-training.csv`) into a DataFrame using Pandas. Using a DataFrame allows us to inspect, clean, and engineer features containing variables of different types for later modelling. 

We also add a `Date` column to our DataFrame so that Python can correctly interpret it as an actual date rather than a meaningless string. This will allow us to organise matches matches in chronological order and create season labels.

In [95]:
# Note for others - input your own path to the data below
data_path = "/users/ahmedelganady/Desktop/Uni/Year 3/COMP0036/Beat the Bookie/Data_Files/"

train_df = pd.read_csv(data_path + "epl-training.csv", parse_dates =['Date'], dayfirst = True)

### Sorting the Data in Chronological Order
The match data provided in (`epl-training.csv`) appears to already be sorted. However, I will sort them again as a precautionary practice.

In [10]:
# Sorts the whole DataFrame by the Date column
train_df.sort_values('Date', inplace=True)

# Resets the old indexing to the new indexing for the sorted data
train_df.reset_index(drop=True, inplace=True)

train_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,2000-08-19,Charlton,Man City,4.0,0.0,H,2.0,0.0,H,Rob Harris,...,14.0,4.0,6.0,6.0,13.0,12.0,1.0,2.0,0.0,0.0
1,2000-08-19,Chelsea,West Ham,4.0,2.0,H,1.0,0.0,H,Graham Barber,...,10.0,5.0,7.0,7.0,19.0,14.0,1.0,2.0,0.0,0.0
2,2000-08-19,Coventry,Middlesbrough,1.0,3.0,A,1.0,1.0,D,Barry Knight,...,3.0,9.0,8.0,4.0,15.0,21.0,5.0,3.0,1.0,0.0
3,2000-08-19,Derby,Southampton,2.0,2.0,D,1.0,2.0,A,Andy D'Urso,...,4.0,6.0,5.0,8.0,11.0,13.0,1.0,1.0,0.0,0.0
4,2000-08-19,Leeds,Everton,2.0,0.0,H,2.0,0.0,H,Dermot Gallagher,...,8.0,6.0,6.0,4.0,21.0,20.0,1.0,3.0,0.0,0.0


### Creating a (Per Match) Goal Difference Feature
The Goal Difference feature will be defined as: 

<center><i>GD = FTHG - FTAG</i></center>

This will quantify how many more or less goals the home team was able to get by the away team. It is important to emphasize that this is a per-match feature and not to be confused with the conventional goal-difference aggregate feature in football league tables.

**Why is this feature useful?**

It will be used in later modelling features such as:
- Quickly interpreting a match's 'intensity'.
- Estimating a team's attack and defense strength in the Poisson GLM.
- Calculating performance a team's performance metrics.

In [12]:
train_df['GD'] = train_df['FTHG'] - train_df['FTAG']

train_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,AST,HC,AC,HF,AF,HY,AY,HR,AR,GD
0,2000-08-19,Charlton,Man City,4.0,0.0,H,2.0,0.0,H,Rob Harris,...,4.0,6.0,6.0,13.0,12.0,1.0,2.0,0.0,0.0,4.0
1,2000-08-19,Chelsea,West Ham,4.0,2.0,H,1.0,0.0,H,Graham Barber,...,5.0,7.0,7.0,19.0,14.0,1.0,2.0,0.0,0.0,2.0
2,2000-08-19,Coventry,Middlesbrough,1.0,3.0,A,1.0,1.0,D,Barry Knight,...,9.0,8.0,4.0,15.0,21.0,5.0,3.0,1.0,0.0,-2.0
3,2000-08-19,Derby,Southampton,2.0,2.0,D,1.0,2.0,A,Andy D'Urso,...,6.0,5.0,8.0,11.0,13.0,1.0,1.0,0.0,0.0,0.0
4,2000-08-19,Leeds,Everton,2.0,0.0,H,2.0,0.0,H,Dermot Gallagher,...,6.0,6.0,4.0,21.0,20.0,1.0,3.0,0.0,0.0,2.0


### Creating a Season Indicator Feature

The data English Premier League provided spans 10+ years, it is important for this project that we can easliy identify which match belongs to which season. For eg: matches between August 2014 and May 2015 belong in the 14/15 season. 

**Why is this an important feature?**

This will allow us to later apply validation techniques such as *Leave One Season Out*, and without a label identifying which matches belong to which season matches from different seasons can be mixed up, affecting the validation process. 

**Season Indicator logic**

The simplest rule would be to say that if a match is in January – June → it belongs to the previous season and if it is in July – December → it belongs to the season starting that year.

In [18]:
# Create a function to assign season label based on match date - in accordance to feature logic explained above.

def season_name(date):
    month = date.month
    year = date.year
    
    if month >= 8:          
        return year 
    else:                   
        return year - 1
    
# Apply the function to DataFrame
train_df['Season'] = train_df['Date'].apply(season_name)

# Confirm
train_df[['Date', 'Season']].head()

,Date,Season
0,2000-08-19,2000.0
1,2000-08-19,2000.0
2,2000-08-19,2000.0
3,2000-08-19,2000.0
4,2000-08-19,2000.0


Run a few sanity checks to ensure everything is working properly.

In [19]:
# Check the total number of seasons
train_df['Season'].unique()

# Count the total number of matches per season 
# Note: each season should show 380 matches
train_df['Season'].value_counts().sort_index()


Season
2000.0    380
2001.0    380
2002.0    380
2003.0    380
2004.0    380
2005.0    380
2006.0    380
2007.0    380
2008.0    380
2009.0    380
2010.0    380
2011.0    380
2012.0    380
2013.0    380
2014.0    380
2015.0    380
2016.0    380
2017.0    380
2018.0    380
2019.0    380
2020.0    380
2021.0    380
2022.0    380
2023.0    480
2024.0    380
Name: count, dtype: int64

## Constructing The Design Matrix For GLM

In this section I will represent the football matches numerically so that they can be used by the Generalised Linear Model (GLM). As this model is fully numerical, we can not give it strings.

**How will the matrix work?**

For every match:
- the home team will be assigned a '+1'.
- the away team will be assigned a '-1'.
- every other team will be assigned a '0'.

### Extract Team List & Create Index Mapping

In order to build the GLM model we must first identify all the unique teams that appear in our data set. We then create a dictionary which maps team names to their unique column indices - so that we can refer to them in the matrix.

Note: This matrix will include all the teams in the dataset, those that got relegated and promoted. Furthermore, I have checked if there is a team which is new to the EPL with no historical data in our dataset - there wasn't. 

In [24]:
# Extracting all team names from dataset
home_team_names = train_df['HomeTeam'].dropna().unique().tolist()

away_team_names = train_df['AwayTeam'].dropna().unique().tolist()

team_names = sorted(list(set(home_team_names+away_team_names))) # used the set function to remove duplicates 

# Create a Dictionary to map unique team names to unique indices
team_index = {}
i = 0

for team in team_names:
    team_index[team] = i
    i += 1

# Check results
print("Number of teams:", len(team_names))
team_index

Number of teams: 46


{'Arsenal': 0,
 'Aston Villa': 1,
 'Birmingham': 2,
 'Blackburn': 3,
 'Blackpool': 4,
 'Bolton': 5,
 'Bournemouth': 6,
 'Bradford': 7,
 'Brentford': 8,
 'Brighton': 9,
 'Burnley': 10,
 'Cardiff': 11,
 'Charlton': 12,
 'Chelsea': 13,
 'Coventry': 14,
 'Crystal Palace': 15,
 'Derby': 16,
 'Everton': 17,
 'Fulham': 18,
 'Huddersfield': 19,
 'Hull': 20,
 'Ipswich': 21,
 'Leeds': 22,
 'Leicester': 23,
 'Liverpool': 24,
 'Luton': 25,
 'Man City': 26,
 'Man United': 27,
 'Middlesbrough': 28,
 'Newcastle': 29,
 'Norwich': 30,
 "Nott'm Forest": 31,
 'Portsmouth': 32,
 'QPR': 33,
 'Reading': 34,
 'Sheffield United': 35,
 'Southampton': 36,
 'Stoke': 37,
 'Sunderland': 38,
 'Swansea': 39,
 'Tottenham': 40,
 'Watford': 41,
 'West Brom': 42,
 'West Ham': 43,
 'Wigan': 44,
 'Wolves': 45}

### Build the GLM Design Matrix & Target Vector
In this section I will convert each football match into numbers, so that I can input them into the model later. We must create the Design Matrix, *X*, where each row represents one match and each column represents a unique team and the Target vector, *y*, which is a list of the goal differences for each match. Note: *X* and *y* should have the same number of rows.

**Design Matrix Logic:**

For every match:
- the home team will be assigned a '+1'.
- the away team will be assigned a '-1'.
- every other team will be assigned a '0'.

The GLM uses *X* and *y* together, and fits the model using a maximum likelihood estimation to calculate a strength value for each team across all matches. 

In [33]:
def glm_data(matches_df, team_names, team_index):
    
    # Removes any empty rows (otherise NaN error)
    matches_df = matches_df.dropna(subset=['HomeTeam', 'AwayTeam'])
    
    # No. of matches & teams
    no_matches = len(matches_df)
    no_teams = len(team_names)
    
    # Create an empty matrix, X
    X = np.zeros((no_matches, no_teams))

    # y is the GD column from the DataFrame
    y = matches_df['GD'].values

    # Ensure index is formatted properly (numbered in order)
    matches_df = matches_df.reset_index(drop=True)

    # Fill X row by row
    for m in range(no_matches):
        # Reads the home and away teams' names
        h_team = matches_df.loc[m, 'HomeTeam']
        a_team = matches_df.loc[m, 'AwayTeam']

        # Find the teams' index number
        h_col = team_index[h_team]
        a_col = team_index[a_team]

        # Sets a value of '+1' for home, and '-1' for away
        X[m, h_col] = 1.0
        X[m, a_col] = -1.0

    return X, y


# Sanity Check
X_all, y_all = glm_data(train_df, team_names, team_index)
print("Design matrix dimensions:", X_all.shape)
print("Target vector dimensions:", y_all.shape)


Design matrix dimensions: (9600, 46)
Target vector dimensions: (9600,)


### Identifiability

This is a unique problem of this feature. Recall that the GLM equation is given by:

$$
\theta_{\text{home}} - \theta_{\text{away}}
$$

The model can only understand the differences between strengths, so any values which keep the differences the same between two teams are equally valid - giving the model infinite strength solutions. 

Example:

Consider teams A & B:
- Strength(A) = 1.5
- Strength(B) = 0.5
- GD = 1.0

Now add '+10' to both A & B's strengths:
- Strength(A) = 11.5
- Strength(B) = 10.5
- GD = 1.0 (stays the same)

These are both equally valid solutions - so how can the model determine which solution is the correct one? The solution to this problem is to choose one team (ideally a mid table team) and set its strength value to equal '0', this becomes the reference team and all other teams' strengths are measured relative to this team's strength. This allows the model to have one unique strength solution per team.

In [34]:
# Choose any team as reference - choosing last team in DataFrame for ease
ref_team = team_names[-1]
print("The Chosen Reference team with strength = 0 is:", ref_team)

# Determine its column index
ref_index = team_index[ref_team]

# Remove that column from our design matrix, X
X_new = np.delete(X_all, ref_index, axis=1)

# Sanity Check
print("Original X dimensions:", X_all.shape)
print("New X dimensions:", X_new.shape)


The Chosen Reference team with strength = 0 is: Wolves
Original X dimensions: (9600, 46)
New X dimensions: (9600, 45)


## Finding a Strength Value For Each Team

This is done by:
- Fitting a linear regression model
- Translating the model's coefficients to team strength values

### Fitting the GLM

So far, I have created a design matrix, *X*, and a goal differences vector, *y*, so we can fit a simple linear regression model to obtain each team's strength.
<br></br>
<center>Predicted GD ≈ $ \theta_{\text{home}} - \theta_{\text{away}} + b$ </center>

where:
- $\theta_{\text{home}}$ is the home team's strength value
- $\theta_{\text{away}}$ is the away team's strength value
- $b$ is the y-intercept of the linear regression model

The model uses maximum likelihood estimation to choose values for all the teams' strengths so that the predicted GD value is as close as possible to the real GD in the data provided.  

We use a Gaussian GLM here because GD values can be estimated with a normal distribution curve over many matches.

In [39]:
# Add an intercept to the reduced design matrix - this is just a column of contants
X_glm = sm.add_constant(X_new)

# Fit a GLM 
model = sm.GLM(y_all, X_glm, family=sm.families.Gaussian())

# Store the results 
results = model.fit()

# Sanity Check - no. of parameters learned by the model
print("Number of parameters learned:", len(results.params))
print(results.params[:10])


Number of parameters learned: 46
[ 0.35143512  1.25278302  0.30668113  0.13324356  0.26173282 -0.14814163
  0.16173282  0.00607259 -0.62494766  0.51109112]


### Assigning Strength Values to Each Team
In the previous step we fit the GLM, the model was able to learn a list of numbers called the *'parameters'*. The first number in that list is the intercept, and the rest of the numbers correspond to the team strength coefficients in order of index, except for the reference team which has been extracted. This team's strength is set as *'0'*. So, a team with a positive strength value is predicted to be stronger than the reference team and a team with a negative value is predicted to be weaker.

So what we need to do is extract the intercept value from the list and assign the remaining parameters (strength coefficients) to their respective teams and store this in a new DataFrame. By the end we should have one strength value per team.

In [43]:
# Defining the intercept parameters from the fitted model
parameters = results.params
intercept = parameters[0]
strength_c = parameters[1:]

# Quick Sanity check
print("Number of team strength coefficients:", len(strength_c))
print("Number of team colums in the reduced design matrix:", X_new.shape[1]) # Both values should be equal

# Create an empty dictionary to place all the teams and strength coefficients in
team_strengths = {}
c_position = 0              # This is to track the position inside the full list of teams

for i in team_names:
    if i == ref_team:
        team_strengths[i] = 0.0
    else: 
        team_strengths[i] = strength_c[c_position]
        c_position += 1

#  Another Sanity Check 3dzzza
print("Number of teams:", len(team_names))
print("Number of strength values:", len(team_strengths))

# Now we convert this dictionary into a DataFrame
teamStrengths_df = pd.DataFrame({
    'Team': list(team_strengths.keys()),
    'Strength Coefficient': list(team_strengths.values())
})

# Sort the teams from highest strength values (strongest) to lowest strength value (weakest)
teamStrengths_df = teamStrengths_df.sort_values('Strength Coefficient', ascending=False).reset_index(drop=True)

# Quick Sanity Check
print(teamStrengths_df)

Number of team strength coefficients: 45
Number of team colums in the reduced design matrix: 45
Number of teams: 46
Number of strength values: 46
                Team  Strength Coefficient
0            Arsenal              1.252783
1          Liverpool              1.250414
2           Man City              1.242620
3            Chelsea              1.218518
4         Man United              1.189340
5          Tottenham              0.759700
6          Brentford              0.511091
7            Everton              0.420689
8          Newcastle              0.352724
9        Aston Villa              0.306681
10          Brighton              0.291545
11         Leicester              0.280069
12         Blackburn              0.261733
13             Leeds              0.230041
14     Middlesbrough              0.204626
15          West Ham              0.196114
16     Nott'm Forest              0.169548
17    Crystal Palace              0.169491
18            Bolton              0.1

## Validation

In the previous section I calculated a single strength value for each team using the entire dataset. Although, this gives a good estimate of the overall team strength, it has a flaw. It uses future matches to describe past matches - this is known as a *'data leakage'*.

For example: if we let the model use results from the 2023/24 season to predict match outcomes of the 2010/11 season, we would be letting the model "see the future" before predicting the present. This is means the model basically cheated by peeking into future results to predict a present match. This would give inflated accuracy numbers as we validate the model, but will fail once we test it on future data (Jan 2026 matches).

Hence, to build a more realistic model, we must ensure that every match only uses match data that was available to it up that point, and nothing beyond it. We do this be generating a feature called *'Time Aware Team Strengths'*. We do this by:

- We sort all the match data by date (and season). (Already Done)
- For a given season, $S_{k}$, we calculate the team strength values using only the seasons before it.
- We then assign this strength, $S_{k}$, to all the matches in that season, $S_{k}$.
- For the first season in the dataset we assign neutral strength values of 0.

This methodology ensures that each match in the dataset is described using only the match data that would have been available up to that point, which should increase predictive accuracy for gameweek 24 (Jan 31st). 

Furthermore, I have decided to weigh team strengths based on recency. This will use the same methodology as before, but it will assign higher weights to more recent seasons, and lower weights to older seasons. I will do this by applying an exponential decay factor of 0.8 per season so that recent seasons exert a greater influence on the GLM fit.

I should then compare models; one which use this recency bias and one which doesn't and evaluate differences in predictive accuracy between the models to determine whether this would improve accuracy or not.

In [66]:
# Obtain a sorted set of all the seasons
# First remove any empty rows
train_df = train_df.dropna(subset=['Season']).reset_index(drop=True)
seasons = sorted(train_df['Season'].unique())

# Create an empty list which will store all team strength values for different seasons
all_strengths = []
decay_factor = 0.8

# Create a for loop to loop through each season one by one

for i, season in enumerate(seasons):
    print("\n\nProcessing season:", season)
    
    prev_seasons = seasons[:i]
    print("Previous season data which is used for this season:", prev_seasons)
    
    # Case 1: No previous data - season 2000
    if not prev_seasons:
        print("There is no past data available for this season, so will assign '0.0' strength value to all teams.")
        for team in team_names:
            all_strengths.append({
                'Season': season,
                'Team': team,
                'Strength Coefficient': 0.0
            })
        continue
    # Case 2: Previous data available - all other seasons
    else:
        prev_df = train_df[train_df['Season'].isin(prev_seasons)].copy()
        print("No. of previous matches used to estimate strength values:", len(prev_df))
    
        # Computing recency-bias by adding higher weights to recent seasons
        # Find the age of the data compared to current season
        prev_df.loc[:, 'Season Age'] = season - prev_df['Season']

        # Calculate the weight of that season
        prev_df.loc[:, 'Weight'] = decay_factor ** prev_df['Season Age']
    
        # Build the design matrix for previous seasons
        X_prev, y_prev = glm_data(prev_df, team_names, team_index)
        print("X_prev dimensions:", X_prev.shape, "\ny_prev length:", len(y_prev))
    
    # Arrange the weights in a vector of same dimensions as y
    weights = prev_df['Weight'].values
    
    # Remove the reference team's column
    ref_index = team_index[ref_team]
    X_prev_red = np.delete(X_prev, ref_index, axis=1)
    print("X_prev_red dimensions:", X_prev_red.shape)
    X_prev_glm = sm.add_constant(X_prev_red)
    
    # Fit the GLM with weights
    model_prev = sm.GLM(
        y_prev,
        X_prev_glm,
        family = sm.families.Gaussian(),
        freq_weights = weights 
    )
    results_prev = model_prev.fit()
    

    # Obtain parameters
    params_prev = results_prev.params
    intercept_prev = params_prev[0]
    teamStrength_coefs_prev = params_prev[1:]
    
    print("Intercept value for previous season's model:", intercept_prev)
    print("No. of team strength coefficients obtained:", len(teamStrength_coefs_prev))
    
    # Redetermine the strengths for all the teams this season
    season_strengths = {}
    pos_coef = 0
    
    for t in team_names:
        if t == ref_team:
            season_strengths[t] = 0.0
        else:
            season_strengths[t] = teamStrength_coefs_prev[pos_coef]
            pos_coef += 1
    
    # Store one row per (Season, Team)
    for t in team_names:
        all_strengths.append({
            'Season': season,
            'Team': t,
            'Strength Coefficient': season_strengths[t]
        })
    



Processing season: 2000.0
Previous season data which is used for this season: []
There is no past data available for this season, so will assign '0.0' strength value to all teams.


Processing season: 2001.0
Previous season data which is used for this season: [2000.0]
No. of previous matches used to estimate strength values: 380
X_prev dimensions: (380, 46) 
y_prev length: 380
X_prev_red dimensions: (380, 45)
Intercept value for previous season's model: 0.4789473684210529
No. of team strength coefficients obtained: 45


Processing season: 2002.0
Previous season data which is used for this season: [2000.0, 2001.0]
No. of previous matches used to estimate strength values: 760
X_prev dimensions: (760, 46) 
y_prev length: 760
X_prev_red dimensions: (760, 45)
Intercept value for previous season's model: 0.37807017543859645
No. of team strength coefficients obtained: 45


Processing season: 2003.0
Previous season data which is used for this season: [2000.0, 2001.0, 2002.0]
No. of previous 

Intercept value for previous season's model: 0.3551235261250582
No. of team strength coefficients obtained: 45


Processing season: 2021.0
Previous season data which is used for this season: [2000.0, 2001.0, 2002.0, 2003.0, 2004.0, 2005.0, 2006.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2012.0, 2013.0, 2014.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0]
No. of previous matches used to estimate strength values: 7980
X_prev dimensions: (7980, 46) 
y_prev length: 7980
X_prev_red dimensions: (7980, 45)
Intercept value for previous season's model: 0.28556249680525847
No. of team strength coefficients obtained: 45


Processing season: 2022.0
Previous season data which is used for this season: [2000.0, 2001.0, 2002.0, 2003.0, 2004.0, 2005.0, 2006.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2012.0, 2013.0, 2014.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0]
No. of previous matches used to estimate strength values: 8360
X_prev dimensions: (8360, 46) 
y_prev length: 8360
X_prev_re

In [84]:
# Add the time-based team-strengths to the main DataFrame

for col in ['HomeTeamStrength', 'AwayTeamStrength', 'StrengthDifference']:
    if col in train_df.columns:
        del train_df[col]

# Convert the all_strengths list into a DataFrame
season_strengths_df = pd.DataFrame(all_strengths)

# Create a home team's strength table - rename the columns
home_strengths_df = season_strengths_df.rename(
    columns={
        'Team': 'HomeTeam',
        'Strength Coefficient': 'HomeTeamStrength'
    }
)

# Add home team strength values into the Data Frame
train_df = train_df.merge(
    home_strengths_df,
    on=['Season', 'HomeTeam'],
    how='left'
)

# Create an away team's strength table - rename the columns
away_strengths_df = season_strengths_df.rename(
    columns={
        'Team': 'AwayTeam',
        'Strength Coefficient': 'AwayTeamStrength'
    }
)

# Add away team strength values into the Data Frame
train_df = train_df.merge(
    away_strengths_df,
    on=['Season', 'AwayTeam'],
    how='left'
)

# Calculate the Strength Difference between the home and away teams
train_df['StrengthDifference'] = train_df['HomeTeamStrength'] - train_df['AwayTeamStrength']

# 7. Sanity checks
print("\nMerged training data with time-aware strengths:")
display(train_df[['Season', 'Date', 'HomeTeam', 'AwayTeam',
                  'HomeTeamStrength', 'AwayTeamStrength', 'StrengthDifference']])

print("\nMissing values check:")
print(train_df[['HomeTeamStrength', 'AwayTeamStrength']].isna().sum())






Merged training data with time-aware strengths:


,Season,Date,HomeTeam,AwayTeam,HomeTeamStrength,AwayTeamStrength,StrengthDifference
0,2000.0,2000-08-19,Charlton,Man City,0.000000,0.000000,0.000000
1,2000.0,2000-08-19,Chelsea,West Ham,0.000000,0.000000,0.000000
2,2000.0,2000-08-19,Coventry,Middlesbrough,0.000000,0.000000,0.000000
3,2000.0,2000-08-19,Derby,Southampton,0.000000,0.000000,0.000000
4,2000.0,2000-08-19,Leeds,Everton,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
9595,2024.0,2025-05-25,Ipswich,West Ham,0.062022,0.154372,-0.092350
9596,2024.0,2025-05-25,Fulham,Man City,-0.029682,1.763041,-1.792723
9597,2024.0,2025-05-25,Bournemouth,Leicester,-0.263437,0.373538,-0.636975
9598,2024.0,2025-05-25,Liverpool,Crystal Palace,1.347241,0.086672,1.260569



Missing values check:
HomeTeamStrength    0
AwayTeamStrength    0
dtype: int64


Now we rename some of the variables to remain consistent with the brief's naming convention.

In [87]:
train_df["HomeStrength"] = train_df["HomeTeamStrength"]
train_df["AwayStrength"] = train_df["AwayTeamStrength"]
train_df["StrengthDiff"] = train_df["HomeStrength"] - train_df["AwayStrength"]

# Sanity Check
train_df[["HomeTeam", "AwayTeam", "HomeStrength", "AwayStrength", "StrengthDiff"]].head()

,HomeTeam,AwayTeam,HomeStrength,AwayStrength,StrengthDiff
0,Charlton,Man City,0.0,0.0,0.0
1,Chelsea,West Ham,0.0,0.0,0.0
2,Coventry,Middlesbrough,0.0,0.0,0.0
3,Derby,Southampton,0.0,0.0,0.0
4,Leeds,Everton,0.0,0.0,0.0


Must check that there are no empty cells (NaN) before passing the data into the model.

In [91]:
# Check if there are any NaNs - How many?
print("No. of NaNs:")
print(train_df[["HomeStrength", "AwayStrength", "StrengthDiff"]].isna().sum())

# Nice - no NaNs

No. of NaNs:
HomeStrength    0
AwayStrength    0
StrengthDiff    0
dtype: int64


## Applying the Time-Based Team Strengths to the Test Data

In this section I will apply the GLM team strength values to the unseen matches. Firstly, I will load the *'epl-test.csv'* file and apply a proper datetime format to the Date column so that it can be comprehended by python. I will then use the season function from before to assign each match to a season.

Then, I will take merge the team strength values that were obtained above onto this data set. I will do this for both the *Home* and *Away* teams - giving every match its own *HomeStrength* and *AwayStrength* columns.

Then, I will create Strength Difference (*StrengthDiff = HomeStrength – AwayStrength*), like I did for the training dataset. The test DataFrame should now be ready for prediction and will contain the 3 final strength columns which the ML model will need.

In [112]:
# Adding the team strength values to the training data
# Load the training data
data_path = "/users/ahmedelganady/Desktop/Uni/Year 3/COMP0036/Beat the Bookie/Data_Files/"
test_df = pd.read_csv(data_path + "epl-test.csv")
test_df["Date"] = pd.to_datetime(test_df["Date"], format="%d %b %y")
test_df["Season"] = test_df["Date"].apply(season_name)

# Clean the data set
test_df["HomeTeam"] = test_df["HomeTeam"].str.strip()
test_df["AwayTeam"] = test_df["AwayTeam"].str.strip()
season_strengths_df["Team"] = season_strengths_df["Team"].str.strip()

# I carried out a sanity check and found that Nottingham forest is called "Nott'm Forest" in the training data
# But, it was called "Nottingham Forest" in the test data
# Here I change the name in the test data so that I can look up its strength and append it to the test DataFrame
forest_name_change = {
    "Nottingham Forest": "Nott'm Forest"
}
test_df["HomeTeam"] = test_df["HomeTeam"].replace(forest_name_change)
test_df["AwayTeam"] = test_df["AwayTeam"].replace(forest_name_change)

# Since this is a new season - no data available from earlier matches this season, will use last season's strengths
prev_season = season_strengths_df["Season"].max()

prev_season_strengths = season_strengths_df[
    season_strengths_df["Season"] == prev_season
].copy()

# Create lookup tables for both home and away teams
home_strengths_df = prev_season_strengths.rename(
    columns={
        "Team": "HomeTeam",
        "Strength Coefficient": "HomeStrength"
    }
)[["HomeTeam", "HomeStrength"]]

away_strengths_df = prev_season_strengths.rename(
    columns={
        "Team": "AwayTeam",
        "Strength Coefficient": "AwayStrength"
    }
)[["AwayTeam", "AwayStrength"]]

# Merge the strengths into the test data
test_df = test_df.merge(home_strengths_df, on="HomeTeam", how="left")
test_df = test_df.merge(away_strengths_df, on="AwayTeam", how="left")

# Create and add the Strength Difference Column to the test data
test_df["StrengthDiff"] = test_df["HomeStrength"] - test_df["AwayStrength"]

# Sanity Check
test_df[["Date", "HomeTeam", "AwayTeam", "Season",
         "HomeStrength", "AwayStrength", "StrengthDiff"]]


,Date,HomeTeam,AwayTeam,Season,HomeStrength,AwayStrength,StrengthDiff
0,2026-01-31,Leeds,Arsenal,2025,-0.203062,1.154965,-1.358027
1,2026-01-31,Liverpool,Newcastle,2025,1.347241,0.419670,0.927571
2,2026-01-31,Tottenham,Man City,2025,0.829967,1.763041,-0.933074
3,2026-01-31,Wolves,Bournemouth,2025,0.000000,-0.263437,0.263437
4,2026-01-31,Aston Villa,Brentford,2025,0.344658,0.315286,0.029372
5,2026-01-31,Brighton,Everton,2025,0.243623,0.082189,0.161433
6,2026-01-31,Chelsea,West Ham,2025,0.799270,0.154372,0.644898
7,2026-01-31,Man United,Fulham,2025,0.678778,-0.029682,0.708460
8,2026-01-31,Sunderland,Burnley,2025,-0.269710,-0.343009,0.073299
9,2026-01-31,Nott'm Forest,Crystal Palace,2025,-0.222569,0.086672,-0.309241


**Note:** The 2025/26 season is defined as 2025 earlier on. Not to be confused with the previous season - 2024/25 (which is *season = 2024*).